# 08. GARCH × Q 추정 방식 비교

## 비교 대상 (5종 + 벤치마크 2)

| 전략 | Q 산출 방식 | 비고 |
|------|-----------|------|
| `Q_FIXED` | 06번 GARCH 최적 Q (자동 로드) | 기준선 |
| `Q_hist` | mean(저위험 − 고위험) 60M window | 학습 기간 평균 실현수익 |
| `Q_momentum` | mean(저위험 − 고위험) 12M window | 단기 추세 |
| `Q_lambda` | Q_FIXED × clip(λ_t / λ_mean, 0.1, 3.0) | 위험회피계수 스케일 |
| `Q_ff3` | FF3 회귀 → P·r̂ | 팩터모델 기반 |
| **CAPM** | π = λΣw_mkt (BL 없음) | 기준선 |
| **SPY** | S&P 500 매수보유 | 시장 벤치마크 |

> Vol 소스는 **GARCH(05번 예측) 단일**. Q 추정 방식만 5가지로 비교한다.
> 06번 민감도 분석에서 Sharpe 최대 Q 값을 자동 로드(`Q_FIXED`)하여 다른 4개 추정 방식과 비교.

## 실행 순서 의존성
```
05_VolatilityPrediction → vol_predicted.csv
06_Q_Sensitivity        → q_sensitivity_stats.csv  (Q_FIXED 자동 로드)
         ↓
08_BL_Q_Methods (GARCH × 5Q 비교)
```

## 기간
- **2011-01 ~ 2025-12** (180개월) — GARCH 예측 시작 시점


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
import platform
from pathlib import Path
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
from bl_utils import *  # 공통 BL/Q/성과 유틸리티

warnings.filterwarnings('ignore')
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
OUT_DIR  = BASE_DIR / 'outputs' / '08_BL_Q_Methods'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 파라미터 ────────────────────────────────────────────────
TRAIN_WINDOW    = 60
TAU             = 0.1
PCT_GROUP       = 0.30
START_PRED      = '2011-01-01'
MOMENTUM_WINDOW = 12
LAM_MEAN        = 2.5

# ── 06번 예측 vol 최적 Q 자동 로드 (GARCH 또는 LSTM 등 활성 모델) ───────────────────────────────
GARCH_STATS = BASE_DIR / 'outputs' / '06_Q_Sensitivity' / 'q_sensitivity_stats.csv'
if GARCH_STATS.exists():
    _gs = pd.read_csv(GARCH_STATS, index_col='label')
    Q_PRED = float(_gs['sharpe'].idxmax().split('=')[1])
    print(f'06번 최적 Q 로드 → Q_PRED = {Q_PRED}')
else:
    Q_PRED = 0.003
    print(f'06번 결과 없음 → 기본값 Q_PRED = {Q_PRED}')
    print('  (06_Q_Sensitivity.ipynb 먼저 실행하면 최적값 자동 적용)')

# ── 데이터 로드 ──────────────────────────────────────────────
panel = pd.read_csv(DATA_DIR / 'monthly_panel.csv', parse_dates=['date'])
panel = panel.set_index(['date', 'ticker'])

all_dates  = panel.index.get_level_values('date').unique().sort_values()
pred_dates = all_dates[all_dates >= START_PRED]

ret_pivot  = panel['ret_1m'].unstack('ticker')
spy_series = panel['spy_ret'].groupby(level='date').first()
rf_series  = panel['rf_1m'].groupby(level='date').first()

print(f'패널: {panel.shape}')
print(f'예측 기간: {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')

In [ ]:
# ── FF3 팩터 로드 ─────────────────────────────────────────────
FF3_PATH = DATA_DIR / 'ff3_monthly.csv'
if FF3_PATH.exists():
    ff3 = pd.read_csv(FF3_PATH, index_col='date', parse_dates=True)
    print(f'FF3 로드: {ff3.shape}')
else:
    print('FF3 다운로드 중...')
    ff3 = download_ff3()
    ff3.to_csv(FF3_PATH)
    print(f'FF3 저장: {ff3.shape}')

# ── GARCH 예측 vol 로드 ──────────────────────────────────────
PRED_PATH = DATA_DIR / 'vol_predicted.csv'
assert PRED_PATH.exists(), '05_VolatilityPrediction.ipynb 먼저 실행해 vol_predicted.csv 생성하세요'
vol_pred_df    = pd.read_csv(PRED_PATH, parse_dates=['date'])
vol_pred_pivot = vol_pred_df.set_index(['date', 'ticker'])['vol_pred'].unstack('ticker')
print(f'GARCH 예측 로드: {vol_pred_pivot.shape}')

In [ ]:
# ── Walk-forward: GARCH × 5Q 비교 ──────────────────────────
results = {f'Q_{m}': [] for m in ['FIXED', 'hist', 'momentum', 'lambda', 'ff3']}
results['CAPM'] = []
results['SPY']  = []
q_log = []

print(f'Walk-forward: {pred_dates[0].date()} ~ {pred_dates[-1].date()} ({len(pred_dates)}개월)')
print(f'  Q_FIXED = Q_PRED = {Q_PRED}  (월 {Q_PRED*100:.1f}%  |  연 {Q_PRED*12*100:.1f}%)')

for i, pred_date in enumerate(pred_dates):
    if (i + 1) % 12 == 0:
        print(f'  {pred_date.date()} ({i+1}/{len(pred_dates)})')
    try:
        if pred_date not in vol_pred_pivot.index:
            continue

        month_base = panel.xs(pred_date, level='date').dropna(
            subset=['vol_21d', 'log_mcap', 'ret_1m'])
        if len(month_base) < 30:
            continue

        pred_vol = vol_pred_pivot.loc[pred_date].reindex(month_base.index)
        month_df = month_base.copy()
        month_df['vol_use'] = pred_vol
        month_df = month_df.dropna(subset=['vol_use'])
        if len(month_df) < 20:
            continue

        idx         = all_dates.get_loc(pred_date)
        train_dates = all_dates[max(0, idx - TRAIN_WINDOW): idx]
        rf_s        = rf_series.reindex(train_dates)
        spy_s       = spy_series.reindex(train_dates)
        spy_excess  = (spy_s - rf_s).mean()
        sigma2_mkt  = spy_s.var()

        universe  = month_df.index.tolist()
        thresh    = int(len(train_dates) * 0.7)
        ret_slice = (ret_pivot.reindex(index=train_dates, columns=universe)
                              .dropna(axis=1, thresh=thresh)
                              .fillna(0))
        valid_tix = ret_slice.columns.tolist()
        if len(valid_tix) < 20:
            continue

        month_df = month_df.reindex(valid_tix)
        Sigma    = compute_sigma(ret_slice)
        mcap     = np.exp(month_df['log_mcap'])
        w_mkt    = (mcap / mcap.sum()).reindex(valid_tix).fillna(0)
        pi, lam  = compute_pi(Sigma, w_mkt, spy_excess, sigma2_mkt)

        P     = build_P(month_df['vol_use'], mcap, pct=PCT_GROUP)
        P     = P.reindex(valid_tix).fillna(0)
        omega = compute_omega(P, Sigma, TAU)

        # ── Q 5종 계산 ──────────────────────────────────────
        q_vals = {
            'FIXED':    Q_PRED,
            'hist':     compute_Q_hist(P, ret_slice),
            'momentum': compute_Q_momentum(P, ret_slice, window=MOMENTUM_WINDOW),
            'lambda':   compute_Q_lambda(lam, Q_PRED, lam_mean=LAM_MEAN),
            'ff3':      compute_Q_ff3(P, ret_slice, ff3.reindex(train_dates), rf_s),
        }
        q_log.append({'date': pred_date, **{f'Q_{k}': v for k, v in q_vals.items()}})

        next_date  = all_dates[idx + 1] if idx + 1 < len(all_dates) else None
        actual_ret = (ret_pivot.loc[next_date].reindex(valid_tix).fillna(0)
                            if next_date is not None
                            else month_df['fwd_ret_1m'].reindex(valid_tix).fillna(0))

        for q_name, q_val in q_vals.items():
            mu_BL = black_litterman(pi, Sigma, P, q_val, omega, TAU)
            w_bl  = optimize_portfolio(mu_BL, Sigma, lam)
            results[f'Q_{q_name}'].append({'date': pred_date, 'ret': float(w_bl @ actual_ret)})

        # 벤치마크: CAPM (BL 없음, π를 그대로 사용)
        w_capm = optimize_portfolio(pi, Sigma, lam)
        results['CAPM'].append({'date': pred_date, 'ret': float(w_capm @ actual_ret)})
        # SPY
        results['SPY'].append({'date': pred_date, 'ret': float(spy_series.get(next_date, np.nan))})

    except Exception as e:
        if i < 5:
            print(f'  [에러] {pred_date.date()}: {e}')

# 결과 시계열 정리
ret_series = {}
for k, v in results.items():
    if v:
        ret_series[k] = pd.DataFrame(v).set_index('date')['ret'].dropna()

q_df = pd.DataFrame(q_log).set_index('date') if q_log else pd.DataFrame()
print(f'\n완료')
for k, s in ret_series.items():
    print(f'  {k}: {len(s)}개월')

In [ ]:
# ── 성과 분석 ────────────────────────────────────────────────
rf_monthly = panel['rf_1m'].groupby(level='date').first()

ORDER = ['Q_FIXED', 'Q_hist', 'Q_momentum', 'Q_lambda', 'Q_ff3', 'CAPM', 'SPY']

stats_list = [performance(ret_series[k], rf_monthly, k) for k in ORDER if k in ret_series]
stats_df   = pd.DataFrame(stats_list).set_index('label')

disp = stats_df.copy()
for col in ['ann_ret', 'ann_vol', 'cum_ret', 'mdd']:
    disp[col] = disp[col].map('{:.2%}'.format)
for col in ['sharpe', 'calmar']:
    disp[col] = disp[col].map('{:.3f}'.format)
disp.columns = ['연환산수익률', '연환산변동성', 'Sharpe', 'Calmar', '누적수익률', 'MDD']

print('=' * 75)
print('GARCH × Q 추정 방식 성과 비교 (2011-01 ~ 2025-12)')
print('=' * 75)
print(disp.to_string())

# Q_FIXED 대비 Sharpe / MDD 차이
base_sh  = stats_df.loc['Q_FIXED', 'sharpe']
base_mdd = stats_df.loc['Q_FIXED', 'mdd']
print(f'\nvs Q_FIXED (Sharpe {base_sh:.3f} | MDD {base_mdd:.2%}):')
for name in ORDER:
    if name in stats_df.index and name != 'Q_FIXED':
        ds = stats_df.loc[name, 'sharpe'] - base_sh
        dm = stats_df.loc[name, 'mdd']    - base_mdd
        print(f'  {name:12s}  ΔSharpe {ds:+.3f}   ΔMDD {dm:+.2%}')

In [ ]:
# ── 시각화 ───────────────────────────────────────────────────
STYLE = {
    'Q_FIXED':    ('#1565C0', '-',  2.0),
    'Q_hist':     ('#64B5F6', '-',  1.5),
    'Q_momentum': ('#00838F', '-',  1.5),
    'Q_lambda':   ('#558B2F', '-',  1.5),
    'Q_ff3':      ('#BBDEFB', '-',  1.5),
    'CAPM':       ('#FF9800', '--', 1.4),
    'SPY':        ('#388E3C', '--', 1.4),
}

fig, axes = plt.subplots(1, 3, figsize=(22, 6.5))
fig.suptitle('GARCH × Q 추정 방식 비교 (2011~2025)', fontsize=13, fontweight='bold')

# ① 누적 수익률
ax = axes[0]
for k in ORDER:
    if k not in ret_series:
        continue
    color, ls, lw = STYLE[k]
    cum = (1 + ret_series[k]).cumprod() - 1
    cum.plot(ax=ax, label=k, color=color, linestyle=ls, linewidth=lw)
ax.set_title('누적 수익률')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(fontsize=8)
ax.axhline(0, color='black', linewidth=0.5)

# ② Sharpe
ax = axes[1]
keys    = [s['label'] for s in stats_list]
sharpes = [s['sharpe'] for s in stats_list]
colors  = [STYLE[k][0] for k in keys]
bars = ax.bar(range(len(keys)), sharpes, color=colors)
ax.set_xticks(range(len(keys)))
ax.set_xticklabels(keys, rotation=30, ha='right', fontsize=8)
ax.set_title('Sharpe Ratio')
ax.axhline(0, color='black', linewidth=0.5)
for bar, val in zip(bars, sharpes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ③ MDD
ax = axes[2]
mdds = [s['mdd'] for s in stats_list]
bars = ax.bar(range(len(keys)), mdds, color=colors)
ax.set_xticks(range(len(keys)))
ax.set_xticklabels(keys, rotation=30, ha='right', fontsize=8)
ax.set_title('MDD')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.axhline(0, color='black', linewidth=0.5)
for bar, val in zip(bars, mdds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.005,
            f'{val:.2%}', ha='center', va='top', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'q_methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 결과 저장
pd.DataFrame(ret_series).to_csv(DATA_DIR / "q_methods_returns.csv")
stats_df.to_csv(OUT_DIR / 'q_methods_stats.csv')
if not q_df.empty:
    q_df.to_csv(OUT_DIR / 'q_methods_log.csv')
print('저장 완료')